# 🧠🤖 每日学习 | 第2周-Day7 🔄

## 📖 今日复习：W1-W2 Transformer核心概念全串联

Hello Jason！今天是复习日，我们一起回顾第一周和第二周学过的Transformer核心知识。🎯

经过14天的学习，你已经掌握了Transformer的基础架构和深度优化！今天我们要把这些知识串起来，形成完整的技术图谱。🔗

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple
import math

# 设置中文字体和样式
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8')

print("🎉 Transformer知识复习系统启动！")

### 🔄 第一周回顾：Transformer基础架构

📝 **Week1核心要点：**
• **Day1**: 自注意力机制 - 每个token关注所有token，计算相关性权重
• **Day2**: Multi-Head Attention - 多个头并行捕捉不同模式的信息
• **Day3**: FFN + LayerNorm + 残差连接 - 深度网络的关键组件
• **Day4**: Tokenizer与词嵌入 - 文本如何变成数字向量
• **Day5**: GPT vs BERT - 两种不同的预训练范式

💡 **关键概念：** 注意力权重、多头并行、位置编码、残差连接

In [ ]:
def self_attention(query: np.ndarray, key: np.ndarray, value: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """实现自注意力机制"""
    d_k = query.shape[-1]
    scores = np.dot(query, key.T) / np.sqrt(d_k)  # 缩放点积注意力
    attention_weights = np.exp(scores) / np.sum(np.exp(scores), axis=-1, keepdims=True)
    output = np.dot(attention_weights, value)
    return output, attention_weights

# 模拟3个token的表示，每个token有4维向量
np.random.seed(42)
tokens = np.random.randn(3, 4)  # (sequence_length, d_model)
output, attn_weights = self_attention(tokens, tokens, tokens)

print("🔍 自注意力机制演示：")
print(f"输入token形状: {tokens.shape}")
print(f"输出形状: {output.shape}")
print(f"注意力权重矩阵:\n{attn_weights:.3f}")

# 可视化注意力权重
plt.figure(figsize=(8, 6))
sns.heatmap(attn_weights, annot=True, cmap='Blues', xticklabels=['Token1', 'Token2', 'Token3'], 
            yticklabels=['Token1', 'Token2', 'Token3'])
plt.title('自注意力权重矩阵', fontsize=14, fontweight='bold')
plt.xlabel('Key tokens')
plt.ylabel('Query tokens')
plt.tight_layout()
plt.show()

### 🔧 第二周回顾：Transformer深度优化

📝 **Week2核心要点：**
• **Day1**: 位置编码 - 让模型理解词序：RoPE、ALiBi
• **Day2**: Transformer推理优化 - KV Cache、Flash Attention、GQA
• **Day3**: QLoRA微调 - 量化微调降低内存需求
• **Day4**: 模型对齐 - RLHF、DPO等人类偏好学习
• **Day5**: SFT全流程 - 从数据准备到训练评估

💡 **关键概念：** 位置编码、KV缓存、Flash Attention、量化微调

In [ ]:
def rotary_position_encoding(seq_len: int, d_model: int) -> np.ndarray:
    """实现RoPE位置编码"""
    positions = np.arange(seq_len)[:, np.newaxis]
    dim = np.arange(d_model)[np.newaxis, :]
    
    # 计算频率
    freq = 1 / (10000 ** (2 * dim / d_model))
    
    # 计算旋转角度
    angle = positions * freq
    
    # 构建旋转矩阵
    cos_angle = np.cos(angle)
    sin_angle = np.sin(angle)
    
    # 生成位置编码
    pos_encoding = np.zeros((seq_len, d_model))
    pos_encoding[:, 0::2] = cos_angle[:, 0::2]  # 偶数维度
    pos_encoding[:, 1::2] = sin_angle[:, 1::2]  # 奇数维度
    
    return pos_encoding

# 演示RoPE位置编码
seq_len, d_model = 8, 64
rope_encoding = rotary_position_encoding(seq_len, d_model)

print(f"🔄 RoPE位置编码形状: {rope_encoding.shape}")
print("前3个token的前8维编码：")
print(rope_encoding[:3, :8])

# 可视化位置编码
plt.figure(figsize=(12, 8))
plt.subplot(2, 2, 1)
plt.imshow(rope_encoding, cmap='RdBu', aspect='auto')
plt.title('RoPE位置编码热力图')
plt.xlabel('维度')
plt.ylabel('位置')

plt.subplot(2, 2, 2)
for i in range(0, seq_len, 2):
    plt.plot(rope_encoding[i, :20], label=f'Position {i}')
plt.title('不同位置的前20维编码')
plt.xlabel('维度')
plt.ylabel('编码值')
plt.legend()

plt.subplot(2, 2, 3)
plt.plot(rope_encoding[:, 0], label='Dim 0')
plt.plot(rope_encoding[:, 1], label='Dim 1')
plt.title('前2维随位置变化')
plt.xlabel('位置')
plt.ylabel('编码值')
plt.legend()

plt.subplot(2, 2, 4)
dim_pairs = [(0, 1), (2, 3), (4, 5)]
for i, (d1, d2) in enumerate(dim_pairs):
    plt.plot(rope_encoding[:, d1], rope_encoding[:, d2], 
             label=f'Dim {d1}-{d2}', marker='o', markersize=3)
plt.title('维度对位置编码轨迹')
plt.xlabel(f'Dim {d1}')
plt.ylabel(f'Dim {d2}')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
def kv_cache_simulation(original_seq_len: int, optimized_seq_len: int) -> dict:
    """模拟KV缓存优化效果"""
    # 假设每个token的key/value向量大小为512维，每个浮点数4字节
    d_kv = 512
    bytes_per_token = 2 * d_kv * 4  # key + value
    
    # 计算内存使用量
    original_memory = original_seq_len * bytes_per_token / (1024**2)  # MB
    optimized_memory = optimized_seq_len * bytes_per_token / (1024**2)  # MB
    
    # 计算加速比（假设推理速度与内存访问成正比）
    speedup = original_memory / optimized_memory
    
    return {
        'original_seq_len': original_seq_len,
        'optimized_seq_len': optimized_seq_len,
        'original_memory_mb': original_memory,
        'optimized_memory_mb': optimized_memory,
        'memory_reduction_mb': original_memory - optimized_memory,
        'speedup_ratio': speedup,
        'compression_ratio': optimized_seq_len / original_seq_len
    }

# 模拟长文本推理中的KV缓存优化
print("🚀 KV缓存优化效果分析：")
scenarios = [
    (1024, 512),   # 从1024个token压缩到512个
    (2048, 768),   # 从2048个token压缩到768个
    (4096, 1024),  # 从4096个token压缩到1024个
]

results = []
for orig, opt in scenarios:
    result = kv_cache_simulation(orig, opt)
    results.append(result)
    print(f"序列长度 {orig}→{opt}: 内存减少 {result['memory_reduction_mb']:.1f}MB, 加速 {result['speedup_ratio']:.2f}x")

# 可视化KV缓存优化效果
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
orig_sizes = [r['original_memory_mb'] for r in results]
opt_sizes = [r['optimized_memory_mb'] for r in results]
x_labels = [f'{orig}→{opt}' for orig, opt in scenarios]

plt.bar([x-0.2 for x in range(len(scenarios))], orig_sizes, width=0.4, label='原始', alpha=0.7)
plt.bar([x+0.2 for x in range(len(scenarios))], opt_sizes, width=0.4, label='优化后', alpha=0.7)
plt.xlabel('场景')
plt.ylabel('内存使用 (MB)')
plt.title('KV缓存内存优化')
plt.xticks(range(len(scenarios)), x_labels, rotation=45)
plt.legend()

plt.subplot(1, 3, 2)
speedups = [r['speedup_ratio'] for r in results]
plt.bar(range(len(scenarios)), speedups, color='green', alpha=0.7)
plt.xlabel('场景')
plt.ylabel('加速倍数')
plt.title('KV缓存加速效果')
plt.xticks(range(len(scenarios)), x_labels, rotation=45)
for i, v in enumerate(speedups):
    plt.text(i, v + 0.1, f'{v:.1f}x', ha='center', va='bottom')

plt.subplot(1, 3, 3)
compression_ratios = [r['compression_ratio'] for r in results]
plt.plot(range(len(scenarios)), compression_ratios, 'o-', linewidth=2, markersize=8, color='red')
plt.xlabel('场景')
plt.ylabel('压缩比')
plt.title('序列长度压缩比')
plt.xticks(range(len(scenarios)), x_labels, rotation=45)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 🔗 W1-W2知识图谱串联

🎯 **核心技术栈完整图景：**

```
输入文本 → Tokenizer → 词嵌入 → 位置编码 → 多头注意力 → FFN → LayerNorm → 残差连接 → 输出
    ↓           ↓         ↓        ↓         ↓        ↓         ↓          ↓
    |           |         |        |         |        |         |          |
    └─┐        └─┐      └─┐     └─┐      └─┐     └─┐      └─┐       └─┐
      │            │        │       │        │        │        │         │
      └─ 处理文本   └─ 数字化  └─ 时间信息 └─ 模式提取 └─ 特征变换 └─ 稳定性   └─ 最终结果
```

💡 **业务关联思考：** 糖水店的销售预测可以用Transformer分析时间序列数据，利用注意力机制发现销量与天气、季节、促销活动的关联模式！

In [ ]:
def transformer_heatmap_layer_connections():
    """可视化Transformer各层之间的连接关系"""
    layers = ['输入', 'Tokenizer', '词嵌入', '位置编码', '多头注意力', 'FFN', 'LayerNorm', '残差连接', '输出']
    
    # 创建连接矩阵
    connections = np.zeros((len(layers), len(layers)))
    
    # 定义主要连接路径
    main_path = [(0,1), (1,2), (2,3), (3,4), (4,5), (5,6), (6,7), (7,8)]
    for i, j in main_path:
        connections[i, j] = 1
        
    # 添加反馈连接（用于学习）
    feedback_connections = [(2,1), (3,2), (4,3), (5,4), (6,5), (7,6), (8,7)]
    for i, j in feedback_connections:
            connections[i, j] = 0.3

    plt.figure(figsize=(12, 10))
    sns.heatmap(connections, annot=True, cmap='YlOrRd', 
                xticklabels=layers, yticklabels=layers,
                linewidths=0.5, linecolor='gray')
    
    plt.title('Transformer层间连接关系图', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('输出层', fontsize=12)
    plt.ylabel('输入层', fontsize=12)
    
    # 添加说明
    plt.figtext(0.02, 0.02, 
                '💡 浅色=主要前向传播  深色=学习反馈连接\n'
                '🎯 右下角=完整Transformer架构实现路径', 
                fontsize=10, ha='left')
    
    plt.tight_layout()
    plt.show()

print("🔗 Transformer完整架构连接关系：")
transformer_heatmap_layer_connections()

### 🧩 复习测试题

✏️ **快速测试（5分钟）：**

❶ 自注意力机制的核心目的是什么？
A. 让每个token只关注自己
B. 让每个token关注所有token的相关性
C. 减少计算复杂度
D. 增加模型参数量

❷ RoPE位置编码的主要优势是什么？
A. 编码长度固定
B. 支持相对位置信息
C. 计算速度最快
D. 存储空间最小

❸ QLoRA微调的主要优势是：
A. 提高模型准确率
B. 减少内存需求
C. 训练速度更快
D. 模型更小

💡 **提示：** 回答后我会帮你批改和解析！

### 📚 知识卡片总结

🎴 **本周精华卡片：**

**卡片1: 注意力机制** 🎯
- 核心：Query-Key-Value 三元组
- 作用：捕捉词与词之间的语义关联
- 公式：Attention(Q,K,V) = softmax(QKᵀ/√dₖ)V

**卡片2: 多头注意力** 🔄
- 并行多个头，捕捉不同模式
- 每个头关注不同语义层面
 最终：Concat(head₁,...,headₕ)Wᵒ

**卡片3: 位置编码** 📍
- RoPE：旋转位置编码，支持相对位置
- ALiBi：线性注意力偏置，无需训练
- 作用：让模型理解词序信息

**卡片4: 推理优化** ⚡
- KV Cache：缓存key-value对，避免重复计算
- Flash Attention：IO感知的注意力计算
- GQA：分组查询注意力，平衡质量与效率

### 🎬 推荐学习资源

📺 **推荐视频：**
- [《Transformer架构详解》](https://www.bilibili.com/video/BV1J54y1Q7Kh) (45分钟)
- [《Attention机制原理与实践》](https://www.bilibili.com/video/BV1GJ411x7h7) (38分钟)
- [《大模型训练优化技术》](https://www.bilibili.com/video/BV1tK4y1x7h8) (52分钟)

📖 **延伸阅读：**
- [《Attention Is All You Need》原始论文](https://arxiv.org/abs/1706.03762)
- [《Flash Attention: Fast and Memory-Efficient Exact Attention》](https://arxiv.org/abs/2205.14135)
- [《QLoRA: Efficient Finetuning of Quantized LLMs》](https://arxiv.org/abs/2305.14314)

💡 **下周预告：** W3 大模型训练全景！
• 预训练：数据、规模与涌现能力
• SFT：监督微调的关键细节
• RLHF：人类反馈强化学习详解
• DPO：直接偏好优化
• ⚡代码实战：HuggingFace TRL微调Qwen2-0.5B

In [ ]:
def generate_learning_progress_chart():
    """生成学习进度可视化"""
    weeks = ['W1', 'W2', 'W3', 'W4', 'W5', 'W6', 'W7', 'W8-12']
    transformer_progress = [100, 100, 80, 0, 0, 0, 0, 0]  # W1-W2完成，W3进行中
    neuroscience_progress = [0, 0, 0, 0, 0, 0, 0, 0]  # 脑科学还未开始
    
    plt.figure(figsize=(12, 8))
    
    # 进度条
    x = np.arange(len(weeks))
    width = 0.35
    
    bars1 = plt.bar(x - width/2, transformer_progress, width, label='大模型学习', 
                    color='skyblue', alpha=0.8, edgecolor='navy')
    bars2 = plt.bar(x + width/2, neuroscience_progress, width, label='脑科学学习', 
                    color='lightcoral', alpha=0.8, edgecolor='darkred')
    
    # 添加数值标签
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                plt.text(bar.get_x() + bar.get_width()/2., height + 2,
                        f'{height}%', ha='center', va='bottom', fontweight='bold')
    
    # 标记当前进度
    current_week = 2  # 当前是第2周
    plt.axvline(x=current_week-0.5, color='gold', linestyle='--', linewidth=3, alpha=0.7)
    plt.text(current_week-0.5, 105, '当前进度', ha='center', va='bottom', 
             fontsize=12, fontweight='bold', color='gold')
    
    plt.xlabel('学习周', fontsize=12)
    plt.ylabel('完成度 (%)', fontsize=12)
    plt.title('12周学习进度追踪', fontsize=16, fontweight='bold')
    plt.xticks(x, weeks)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 110)
    
    # 添加阶段说明
    phase_text = [
        'Transformer基础',
        'Transformer深入',
        '大模型训练',
        'RAG与检索',
        '推理与思维链',
        'Agent与工具',
        '多模态与安全',
        '脑科学基础'
    ]
    
    for i, (week, phase) in enumerate(zip(weeks, phase_text)):
        plt.text(i, -10, phase, ha='center', va='top', fontsize=9, rotation=45)
    
    plt.tight_layout()
    plt.show()

print("📊 你的学习进度追踪：")
generate_learning_progress_chart()

print("\n🎉 恭喜Jason！你已经完成了：")
print("✅ W1: Transformer基础架构 (Day1-5)")
print("✅ W2: Transformer深入优化 (Day1-6)")
print("🔄 W3: 大模型训练全景 (即将开始)")
print("📈 总体进度：2/12周，完成16.7%！")

print("\n💪 继续加油，接下来的大模型训练内容更加精彩！")